# Hidden Markov Model
Hidden Markov Models (HMMs) contain hidden states that we are trying to infer from observed data. This is useful in bioinformatics, because the observed data is what we can directly measure, like sequenced DNA. On the other hand, the hidden states represent the underlying biological context we are trying to uncover or infer. We will use the Viterbi algorithm to find the most likely sequence of hidden states given a sequence of observations.

The Baum-Welch algorithm represents an Expectation-Maximization approach to learning HMM parameters, offering several advantages over manual parameter setting:

Key Features

* Unsupervised Learning: Estimates model parameters without labeled training data
* Maximum Likelihood: Finds parameters that maximize observation probability
* Iterative Refinement: Progressively improves parameter estimates
* Convergence Guarantees: Always reaches a local optimum of the likelihood function

*
* Algorithm Structure

The Baum-Welch algorithm involves these key steps:

**Initialization:**
* Start with initial guesses for transition, emission, and initial probabilities
* Set up convergence criteria and pseudocounts to prevent zero probabilities

**Expectation Step (E):**
* Run Forward-Backward algorithm on training sequences
* Calculate expected counts for transitions and emissions
* Compute posterior probabilities for each state at each position
* These expected counts represent how often each transition and emission is used to generate the observed sequences

**Maximization Step (M):**
* Update model parameters based on expected counts
* Re-estimate initial, transition, and emission probabilities using the expected counts
* Normalize to ensure valid probability distributions
* Scale values to prevent numerical underflow

**Iteration and Convergence:**
* Repeat the E and M steps until convergence criteria are met
* Monitor likelihood improvement between iterations
* Handle multiple observation sequences appropriately


In [1]:
import numpy as np
import math
from hmm_utils import ForwardBackward
from project10.project10_vic import emission_probs

In [18]:
class BaumWelch():
    def __init__(self, num_states, seed, pseudocount=1e-14, num_sweeps=10000, convergence_threshold = 1e-6):
        self.num_states = num_states
        self.states = [f"S{i}" for i in range(num_states)]
        self.seed = seed
        self.nucleotide_map = {"A": 0, "C": 1, "G": 2, "T": 3}
        self.num_sweeps = num_sweeps
        self.pseudocount = pseudocount
        self.initial_probs, self.transition_probs, self.emission_probs = self.bw_initialization()
        self.emission_counts = np.full((self.num_states, 4), -np.inf)
        self.transition_counts = np.full((self.num_states, self.num_states), -np.inf)
        self.initial_counts = np.full((self.num_states,), -np.inf)
        self.total_seq_prob = -np.inf
        self.convergence_threshold = convergence_threshold



    def bw_initialization(self):
        # initialize random seed
        rand = np.random.default_rng(self.seed)

        # initial probs
        initial_probs = rand.dirichlet(np.ones(self.num_states))
        initial_probs += self.pseudocount
        initial_probs = np.log(initial_probs)
        #print(f"Initial probabilities:\n {initial_probs}\n")

        # transmission probs
        transition_probs = rand.dirichlet(np.ones(self.num_states), size=self.num_states)
        transition_probs += self.pseudocount
        transition_probs = np.log(transition_probs)
        #print(f"Transmission probabilities:\n {transition_probs}\n")

        # emission probs
        emission_probs = rand.dirichlet(np.ones(len(self.nucleotide_map.keys())), size=self.num_states)
        emission_probs += self.pseudocount
        emission_probs = np.log(emission_probs)
        #print(f"Emission probabilities:\n {emission_probs}\n")

        return initial_probs, transition_probs, emission_probs

    def forward_matrix(self, observations):
        """
        The algorithm computes the forward probability matrix in log-space. Each entry represents the probability of having emitted the observed symbols up to the particular position and state.

        :param observations: list of observed sequences
        :return:
        prob_matrix: np.ndarray - Forward probability matrix of shape (num_states x len(observations)) in log space
        """
        prob_matrix = np.full((self.num_states, len(observations)), -np.inf)

        ####### Iteration ########
        for i, observation in enumerate(observations):
            # calculate probabilities
            # Remember log(x*y) = log(x) + log(y)
            # first column
            if i == 0:
                prob_matrix[:, i] = self.initial_probs + self.emission_probs[:, self.nucleotide_map[observation]]
            else:
                prev = prob_matrix[:, i-1][:, np.newaxis] + self.transition_probs
                prob_matrix[:, i] = np.logaddexp.reduce(prev, axis = 0) + self.emission_probs[:, self.nucleotide_map[observation]]

        return prob_matrix

    def backward_matrix(self, observations):
        """
        The algorithm computes the backward probability matrix. Each entry represents the probability of having emitted the observed symbols from the current positon to the end of the sequence given a specific state.

        :param observations: list of observed sequences
        :return:
        prob_matrix: np.ndarray - Forward probability matrix of shape (num_states x len(observations)) in log space
        """
        reversed_obs = observations[::-1]
        prob_matrix = np.zeros((self.num_states, len(observations)), dtype=float)

        for i, observation in enumerate(reversed_obs):
            #for j, state in enumerate(self.states):
            if i == 0:
                continue
            else:
                prev = prob_matrix[:, i-1][np.newaxis, :] + self.transition_probs

                prob_matrix[:, i] = np.logaddexp.reduce(prev, axis = 1) + self.emission_probs[:, self.nucleotide_map[reversed_obs[i-1]]]


        return np.fliplr(prob_matrix)

    def sequence_probability(self, prob_matrix):
        """
        Computes the total probability of the observation sequence from either the forward probability matrix.

        :param prob_matrix:  np.ndarray - Forward probability matrix of shape (num_states x len(observations))
        :return:
            float - Log probability of the observation sequence
        """
        return np.logaddexp.reduce(prob_matrix[:, -1])

    def forward_backward(self, observations):
        """
        The algorithm computes the forward-backward posterior matrix by combining the forward and backward matrices, normalized by the total probability of the observation sequence.

        :param observations: list of observed sequences
        :return:
        posterior_matrix: np.ndarray - posterior probability matrix of shape (num_states x len(observations)) in log space
        """
        self.posterior_matrix = np.zeros((self.num_states, len(observations)), dtype=float)

        forward_matrix = self.forward_matrix(observations)
        backward_matrix = self.backward_matrix(observations)

        final_col_prob = self.sequence_probability(forward_matrix)

        for i in range(len(observations)):
            for j in range(self.num_states):
                self.posterior_matrix[j][i] = forward_matrix[j][i] + backward_matrix[j][i]

        return forward_matrix, backward_matrix, self.posterior_matrix
    def expectation(self, observations):
        """
        # obs = ["GGCACTGAA", "ATGCAATGC", "AATGCCTGA"]
        total_prob = 0
        for observation in observations:
            forward = Build forward_matrix(observation)
            backward = Build backward_matrix(observation)
            posterior = forward_backward(observations)
            xi = compute_transition_posterior(forward, backward, observation)
            accumulate emission counts (posterior, sequence)
            accumulate transition counts (xi, sequence)
            accumulate initial counts (posterior)
        """

        for obs in observations:
            # Create intermediate matrices
            forward, backward, posterior = self.forward_backward(obs)
            transition_posterior = self.transition_posterior_matrix(forward, backward, obs)

            # Accumulation
            self.accumulate_emission_counts(posterior, obs)
            self.accumulate_transition_counts(transition_posterior)
            self.accumulate_initial_counts(posterior)
            self.accumulate_total_seq_prob(forward)
        #print(f"Emission Counts: {self.emission_counts}")
        #print(f"Transition Counts: {self.transition_counts}")
        #print(f"Initial Counts: {self.initial_counts}")
        #print(f"Total Seq Prob: {self.total_seq_prob}")

    def transition_posterior_matrix(self, forward, backward, observation):
        # Create a 3D matrix filled with -inf (log-space)
        trans_posterior = np.full((self.num_states,len(observation)-1, self.num_states), -np.inf)

        for n in range(len(observation)-1):
            # Vectorization
           trans_posterior[:, n] = forward[:, n][:, np.newaxis] + self.transition_probs + self.emission_probs[:, self.nucleotide_map[observation[n+1]]] + backward[:, n+1][np.newaxis, :]

        # Collapse transition posterior into a 2d array
        trans_posterior = np.logaddexp.reduce(trans_posterior, axis =1)

        return trans_posterior

    def accumulate_emission_counts(self, posterior, sequence):
        for i, pos in enumerate(sequence):
            pos_index = self.nucleotide_map[pos]

            self.emission_counts[:, pos_index] = np.logaddexp(self.emission_counts[:, pos_index], posterior[:, i])

    def accumulate_transition_counts(self, trans_posterior):
        self.transition_counts = np.logaddexp(self.transition_counts, trans_posterior)

    def accumulate_initial_counts(self, posterior):
       self.initial_counts = np.logaddexp(self.initial_counts, posterior[:, 0])

    def accumulate_total_seq_prob(self, forward):
        self.total_seq_prob = np.logaddexp(self.total_seq_prob, self.sequence_probability(forward))

    def normalization(self):
        # Normalize initial counts by total probability of accumulated sequences
        normalized_initial_counts = np.logaddexp(self.initial_counts, np.log(self.pseudocount)) - self.total_seq_prob

        # Normalize emission counts
        posterior_probs_sum_by_states = np.logaddexp.reduce(self.emission_counts, axis=1)
        normalized_emission_counts = np.logaddexp(self.emission_counts, np.log(self.pseudocount)) - posterior_probs_sum_by_states[:, np.newaxis]
        #normalized_emission_counts = self.emission_counts - posterior_probs_sum_by_states[:, np.newaxis]
        #print(f"PC: {self.pseudocount}")

        # Normalize transition counts
        trans_posterior_probs_sum_by_states = np.logaddexp.reduce(self.transition_counts, axis=(1))
        normalized_transition_counts = np.logaddexp(self.transition_counts, np.log(self.pseudocount)) - trans_posterior_probs_sum_by_states

        return normalized_initial_counts, normalized_emission_counts, normalized_transition_counts

    def maximization(self):
        normalized_initial_counts, normalized_emission_counts, normalized_transition_counts = self.normalization()

        #print(f"Norm Emission: {normalized_emission_counts}")
        #print(f"Norm Transition: {normalized_transition_counts}")
        #print(f"Norm Initial: {normalized_initial_counts}\n")
        #print(f"Emission: {np.exp(normalized_emission_counts)}")
        #print(f"Transition: {np.exp(normalized_transition_counts)}")
        #print(f"Initial: {np.exp(normalized_initial_counts)}")
        # Build the new emission probability
        self.emission_probs = normalized_emission_counts

        # Build the new transitions probability
        self.transition_probs = normalized_transition_counts

        # Build the new initial probability
        self.initial_probs = normalized_initial_counts

    def baum_welch(self, observations):
        prev_total_seq_prob = None
        for i in range(self.num_sweeps):
            self.expectation(observations)
            self.maximization()


            if i%10 == 0:

                print(f"\rConvergence check Previous total seq prob: {prev_total_seq_prob} Total seq prob: {self.total_seq_prob}", end="", flush=True)
                if prev_total_seq_prob is not None:
                    convergence_check = math.isclose(self.total_seq_prob, prev_total_seq_prob, abs_tol=self.convergence_threshold)
                    if convergence_check:
                        return f"Convergence reached at {i+1} number of sweeps."

                prev_total_seq_prob = self.total_seq_prob

            self.emission_counts = np.full((self.num_states, 4), -np.inf)
            self.transition_counts = np.full((self.num_states, self.num_states), -np.inf)
            self.initial_counts = np.full((self.num_states,), -np.inf)
            self.total_seq_prob = -np.inf
        return f"Convergence not reached at {i+1} number of sweeps."




In [41]:
test = BaumWelch(num_states=2, seed=42, num_sweeps=150000, convergence_threshold=1e-9)
test.baum_welch(sequences)



Convergence check Previous total seq prob: -37.58696745155424 Total seq prob: -37.58696745155656

array([[5.78472995e+235, 5.78472995e+235, 5.78472995e+235,
        5.78472995e+235],
       [7.20808789e+000, 7.26799488e+000, 7.40513352e+000,
        7.22169370e+000]])

In [29]:
import numpy as np

# States
states = ["GC", "BG"]

# Initial probabilities
initial_probs = {"GC": 0.3, "BG": 0.7}

# Transition probabilities
transition_probs = {
    "GC": {"GC": 0.8, "BG": 0.2},
    "BG": {"GC": 0.1, "BG": 0.9}
}

# Emission probabilities
emission_probs = {
    "GC": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "BG": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}
}

def generate_sequence(length=30, seed=None):
    if seed is not None:
        np.random.seed(seed)

    nucleotides = ["A", "C", "G", "T"]

    # pick initial state
    state = np.random.choice(states, p=[initial_probs["GC"], initial_probs["BG"]])

    seq = []
    hidden = []
    for _ in range(length):
        probs = [emission_probs[state][n] for n in nucleotides]
        nuc = np.random.choice(nucleotides, p=probs)
        seq.append(nuc)
        hidden.append(state)

        trans = transition_probs[state]
        state = np.random.choice(states, p=[trans["GC"], trans["BG"]])

    return "".join(seq), hidden



In [30]:
sequences = []
true_states = []
for i in range(30):
    seq, states_seq = generate_sequence(length=30, seed=i)
    sequences.append(seq)
    true_states.append(states_seq)

In [36]:
# Print sequences
print(sequences)


['TGGTCGTAGGTTGTCTGGGGCAGACCAAAA', 'TCACGGGGGCTGTAGCGCTGTTCAAGCACG', 'ACCGAGGTCAAAGACACCGTGCCTTATAAT', 'TGTACCCGGCGCGCTGTAGACATACAACTG', 'GTAACATGCTTGACCTCGGGGCGCGCAACC', 'GTGGCGCAAGAAATACGGGTGTCCGCACGG', 'CAGCGTGTCTGTGTATGTCCTTCATACTCG', 'GGGACTATCATGATCCCTCACCTTTCGAGG', 'TGACCGTCTATCCGTCGCAGGAGGCGCCTC', 'GCCCCGAGGTGCTATTATTCCCATAGACCC', 'ATATATGGATAGCGGGCCGGGGGGCAATCA', 'AGCCTATGCCGCAGCGGTCTCCCTGCCAGG', 'GGTAAGTGCCACGGGCTTGCTTCCAGACGA', 'ATCTTCGGCCATAGGAGTCCGAAGAGAAGG', 'TATCACAACTACAGCAATTATATCTAAAGT', 'ACGCCATAACCCTCGGCGCCAGCACCGTCA', 'GACCTACCCCTCAGGGGCGAGGCCAAAGTT', 'GAGGCATGGCGGCACGACCGAGTATACCTG', 'GATTATAATGCTTGTATAGGGACAATTAGA', 'GCAGGGACGGTGGTTGGCATATCGTGTGGA', 'TTGGCGGAGCTACTCGACGCGCGGACTGGG', 'CAAGGGCGTCGGTAACATAGCGCTTCTCGC', 'CGCGGGTATAGGGATTCTAGAAGACTCTAC', 'TAGCCGCTAAGAACCTGACGTGACTGTATA', 'GATCCCTAGTCGTGATGCGAGAGCTATATC', 'GAACCAGGGTGTGGCTCGGTGGTGGGCGGC', 'GTACAGTATCGAGATTGGACTCTATATTGG', 'TTTAGTTCTACCGGGGTCGCGATCTGTCAA', 'GCGTTTATAGCGTAATCATAGTAGACACAG', 'AGGGGGCCGTTT